# YgoMaster-PackBuilder — YgoMaster Build Writer Tests

Test offline di `write_ygomaster_build()`.

Verifica:

- creazione della build da zero;
- struttura delle directory;
- contenuto identico agli exporter puri;
- JSON UTF-8 valido;
- overwrite degli artifact posseduti dal writer;
- preservazione di file estranei;
- assenza di file `.tmp` residui;
- errore se `build_dir` è un file;
- fallimento prima della scrittura se gli exporter rifiutano l'input.

Tutto viene eseguito in directory temporanee: la vera `build/` della repository non viene toccata.


## 1. Import e helper

In [1]:
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory
import json

from ygomaster_packbuilder.domain import (
    Card,
    OpeningPolicy,
    PackDefinition,
    PackType,
    ProjectDefinition,
    Rarity,
    RarityDistribution,
    SourcePackRef,
    SourceProvider,
    TenPackGuarantee,
    UltraRarePity,
)

from ygomaster_packbuilder.exporters.ygomaster import (
    YgoMasterBuildArtifacts,
    build_main_shop_settings_patch,
    build_pack_odds_export,
    build_shop_export,
    write_ygomaster_build,
)


def make_policy():
    return OpeningPolicy(
        key="configuration-a",
        normal_slot_distribution=RarityDistribution(
            normal=55.0,
            rare=35.0,
            super_rare=7.5,
            ultra_rare=2.5,
        ),
        ten_pack_guarantee=TenPackGuarantee(
            enabled=True,
            final_slot_distribution=RarityDistribution(
                normal=0.0,
                rare=0.0,
                super_rare=80.0,
                ultra_rare=20.0,
            ),
        ),
        ultra_rare_pity=UltraRarePity(
            enabled=True,
            guaranteed_slot_distribution=RarityDistribution(
                normal=0.0,
                rare=0.0,
                super_rare=0.0,
                ultra_rare=100.0,
            ),
        ),
    )


def make_card(name, ydk_id, md_id, rarity):
    return Card(
        name=name,
        ydk_id=ydk_id,
        source_rarity=rarity.value,
        master_duel_id=md_id,
        rarity=rarity,
    )


def make_pack(
    key,
    name,
    description,
    pack_type,
    booster_size,
    cards,
):
    expected_count = {
        PackType.PERSONAL_MAIN: 100,
        PackType.PERSONAL_MINI: 20,
        PackType.SHARED_STAPLES: 200,
    }[pack_type]

    return PackDefinition(
        key=key,
        name=name,
        description=description,
        type=pack_type,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key,
        ),
        expected_card_count=expected_count,
        booster_size=booster_size,
        opening_policy_key="configuration-a",
        cards=cards,
    )


project = ProjectDefinition(
    name="Writer Test",
    packs=[
        make_pack(
            "main-pack",
            "Drago Élite",
            "Descrizione con caratteri UTF-8: è, à, Ω",
            PackType.PERSONAL_MAIN,
            7,
            [
                make_card("Carta Normale", 1, 1001, Rarity.N),
                make_card("Carta Ultra", 2, 1002, Rarity.UR),
            ],
        ),
        make_pack(
            "mini-pack",
            "Mini Pack",
            "",
            PackType.PERSONAL_MINI,
            3,
            [
                make_card("Carta Rare", 3, 1003, Rarity.R),
                make_card("Carta Super", 4, 1004, Rarity.SR),
            ],
        ),
    ],
    opening_policies=[make_policy()],
)

pack_ids = {
    "main-pack": 9_000_000,
    "mini-pack": 9_000_001,
}

print("Fixtures ready")

Fixtures ready


## 2. Build da zero

In [2]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    assert isinstance(
        artifacts,
        YgoMasterBuildArtifacts,
    )

    assert artifacts.root == build_dir

    assert artifacts.shop == (
        build_dir
        / "Shop"
        / "ygomaster-packbuilder.json"
    )

    assert artifacts.shop_pack_odds == (
        build_dir
        / "ShopPackOdds"
        / "ygomaster-packbuilder.json"
    )

    assert artifacts.shop_settings_patch == (
        build_dir
        / "ShopSettings.patch.json"
    )

    assert artifacts.shop.is_file()
    assert artifacts.shop_pack_odds.is_file()
    assert artifacts.shop_settings_patch.is_file()

    print("Build structure OK")

Build structure OK


## 3. I file contengono esattamente gli exporter puri

In [3]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    actual_shop = json.loads(
        artifacts.shop.read_text(
            encoding="utf-8"
        )
    )

    actual_odds = json.loads(
        artifacts.shop_pack_odds.read_text(
            encoding="utf-8"
        )
    )

    actual_patch = json.loads(
        artifacts.shop_settings_patch.read_text(
            encoding="utf-8"
        )
    )

    expected_shop = build_shop_export(
        project,
        pack_ids,
    )

    expected_odds = build_pack_odds_export(
        project,
        pack_ids,
    )

    expected_patch = (
        build_main_shop_settings_patch()
    )

    assert actual_shop == expected_shop
    assert actual_odds == expected_odds
    assert actual_patch == expected_patch

    print("Written JSON matches pure exporters")

Written JSON matches pure exporters


## 4. UTF-8 preservato

In [4]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    raw = artifacts.shop.read_text(
        encoding="utf-8"
    )

    assert "Drago Élite" in raw
    assert "è, à, Ω" in raw

    # ensure_ascii=False: i caratteri non devono
    # essere trasformati in escape Unicode.
    assert "\\u00c9" not in raw
    assert "\\u03a9" not in raw

    print("UTF-8 output OK")

UTF-8 output OK


## 5. Nessun `.tmp` residuo

In [5]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    tmp_files = [
        path
        for path in build_dir.rglob("*")
        if path.is_file()
        and path.name.endswith(".tmp")
    ]

    assert tmp_files == []

    print("No temporary files left behind")

No temporary files left behind


## 6. Overwrite degli artifact esistenti

In [6]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    artifacts.shop.write_text(
        "STALE SHOP",
        encoding="utf-8",
    )

    artifacts.shop_pack_odds.write_text(
        "STALE ODDS",
        encoding="utf-8",
    )

    artifacts.shop_settings_patch.write_text(
        "STALE PATCH",
        encoding="utf-8",
    )

    new_artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    assert json.loads(
        new_artifacts.shop.read_text(
            encoding="utf-8"
        )
    ) == build_shop_export(
        project,
        pack_ids,
    )

    assert json.loads(
        new_artifacts.shop_pack_odds.read_text(
            encoding="utf-8"
        )
    ) == build_pack_odds_export(
        project,
        pack_ids,
    )

    assert json.loads(
        new_artifacts.shop_settings_patch.read_text(
            encoding="utf-8"
        )
    ) == build_main_shop_settings_patch()

    print("Owned artifacts overwritten correctly")

Owned artifacts overwritten correctly


## 7. File estranei non vengono eliminati

In [7]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    unrelated_root = build_dir / "keep-me.txt"
    unrelated_nested = (
        build_dir
        / "Shop"
        / "other-mod.json"
    )

    unrelated_root.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    unrelated_nested.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    unrelated_root.write_text(
        "root sentinel",
        encoding="utf-8",
    )

    unrelated_nested.write_text(
        "nested sentinel",
        encoding="utf-8",
    )

    write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    assert unrelated_root.read_text(
        encoding="utf-8"
    ) == "root sentinel"

    assert unrelated_nested.read_text(
        encoding="utf-8"
    ) == "nested sentinel"

    print("Unrelated files preserved")

Unrelated files preserved


## 8. `build_dir` esistente ma non directory

In [8]:
with TemporaryDirectory() as tmp:
    build_path = Path(tmp) / "build"

    build_path.write_text(
        "I am a file",
        encoding="utf-8",
    )

    try:
        write_ygomaster_build(
            project,
            pack_ids,
            build_path,
        )
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Un build path che è un file "
            "avrebbe dovuto essere rifiutato."
        )

    assert build_path.read_text(
        encoding="utf-8"
    ) == "I am a file"

print("Non-directory build path rejected")

Expected ValueError: Build path is not a directory: C:\Users\LMDP1\AppData\Local\Temp\tmp4u3_uet1\build
Non-directory build path rejected


## 9. Input invalido → nessun artifact scritto

Gli exporter vengono eseguiti prima delle scritture. Quindi un errore di export deve lasciare la build non creata.


In [9]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    incomplete_ids = {
        "main-pack": 9_000_000,
        # mini-pack volutamente assente
    }

    try:
        write_ygomaster_build(
            project,
            incomplete_ids,
            build_dir,
        )
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Pack ID mancante non rifiutato."
        )

    assert not build_dir.exists()

print("Exporter failure occurs before filesystem writes")

Expected ValueError: No YgoMaster pack ID assigned to pack 'mini-pack'.
Exporter failure occurs before filesystem writes


## 10. Input invalido: opening policy sconosciuta

In [10]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    bad_project = deepcopy(project)

    bad_project.packs[0].opening_policy_key = (
        "missing-policy"
    )

    try:
        write_ygomaster_build(
            bad_project,
            pack_ids,
            build_dir,
        )
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Opening policy sconosciuta "
            "non rifiutata."
        )

    assert not build_dir.exists()

print("Policy failure occurs before filesystem writes")

Expected ValueError: Pack 'main-pack' references unknown opening policy 'missing-policy'.
Policy failure occurs before filesystem writes


## 11. JSON formattato e newline finale

In [11]:
with TemporaryDirectory() as tmp:
    build_dir = Path(tmp) / "build"

    artifacts = write_ygomaster_build(
        project,
        pack_ids,
        build_dir,
    )

    for path in (
        artifacts.shop,
        artifacts.shop_pack_odds,
        artifacts.shop_settings_patch,
    ):
        text = path.read_text(
            encoding="utf-8"
        )

        assert text.endswith("\n")
        assert "\n  " in text

    print("Pretty JSON + final newline OK")

Pretty JSON + final newline OK
